In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cortex

# ==========================================
# 1. Configuration & Toggles
# ==========================================
ROI_A = "A5"
ROI_B = "FFC"
MODE = "group_average" # or "per_subject"

# --- AESTHETIC CONTROLS ---
SHOW_BORDERS = False      # Set to True if Inkscape is installed
VMAX_PERCENTILE = 95      # Matches Hedger's dynamic contrast
CMAP_1D = "plasma"        # Hedger's 1D colormap
CMAP_2D = "nipy_spectral_alpha" # Hedger's 2D colormap

# --- PATHS ---
DATA_BASE = "/home/amin/Research/Representation/Movie/data"
OUT_BASE = "/home/amin/Research/Representation/Movie/outputs/cf_modeling"
PYCORTEX_STORE = f"{DATA_BASE}/hedger2026"
TEMPLATE_CIFTI = "/home/amin/Research/Representation/Movie/data/preprocessed/average_sub/sg_psc/group_average_sg_psc_cortex_59k.dtseries.nii"

if MODE == "group_average":
    prep_dir = f"{OUT_BASE}/{MODE}/{ROI_A}_{ROI_B}/prep"
else:
    prep_dir = f"{OUT_BASE}/{MODE}/{ROI_A}_{ROI_B}/group"

# ==========================================
# 2. Pycortex Initialization (The Singleton Fix)
# ==========================================
new_db = cortex.database.Database(PYCORTEX_STORE)
cortex.db = new_db
cortex.database.default_filestore = new_db
for mod_name, mod in sys.modules.items():
    if mod_name.startswith("cortex") and hasattr(mod, "db"):
        mod.db = new_db
cortex.options.config.set("basic", "filestore", PYCORTEX_STORE)
cx_sub = list(cortex.db.subjects.keys())[0]
print(f"Locked onto Pycortex Anatomy: {cx_sub}")

# ==========================================
# 3. Data Loading & Proper Inflation
# ==========================================
# Load the null-corrected maps and the product map
r2_a_nc = np.load(os.path.join(prep_dir, f"R2_{ROI_A}_nc{'_avg' if MODE=='per_subject' else ''}.npy"))
r2_b_nc = np.load(os.path.join(prep_dir, f"R2_{ROI_B}_nc{'_avg' if MODE=='per_subject' else ''}.npy"))
product_map = np.load(os.path.join(prep_dir, f"product_map{'_avg' if MODE=='per_subject' else ''}.npy"))

bm_axis = nib.load(TEMPLATE_CIFTI).header.get_axis(1)

def inflate_to_hedger_space(cifti_data, axis):
    """
    Inflates 59k CIFTI to 118k Pycortex space.
    CRITICAL: Pads with np.nan to make the medial wall transparent.
    CRITICAL: Offsets the right hemisphere by 59292 to prevent overlapping.
    """
    # Initialize full sphere with NaNs (Fixes the Blue Blob)
    full_cortex = np.full(118584, np.nan, dtype=np.float32) 
    
    for name, data_indices, model in axis.iter_structures():
        if name == "CIFTI_STRUCTURE_CORTEX_LEFT":
            full_cortex[model.vertex] = cifti_data[data_indices]
        elif name == "CIFTI_STRUCTURE_CORTEX_RIGHT":
            # Offset by the exact size of the Left Hemisphere mesh (Fixes Overlap)
            full_cortex[model.vertex + 59292] = cifti_data[data_indices]
            
    return full_cortex

# Clip negative noise to 0, then inflate
r2_a_full = inflate_to_hedger_space(np.clip(r2_a_nc, 0, None), bm_axis)
r2_b_full = inflate_to_hedger_space(np.clip(r2_b_nc, 0, None), bm_axis)
product_full = inflate_to_hedger_space(np.clip(product_map, 0, None), bm_axis)

# ==========================================
# 4. Plotting Functions
# ==========================================
# ==========================================
# 4. Plotting Functions (Corrected)
# ==========================================
def plot_1d_map(data, title):
    print(f"Generating 1D Map: {title}...")
    vmax = np.nanpercentile(data[data > 0], VMAX_PERCENTILE) if np.any(data > 0) else 0.1
    vx = cortex.Vertex(data, subject=cx_sub, vmin=0, vmax=vmax, cmap=CMAP_1D)
    
    fig = cortex.quickflat.make_figure(
        vx, 
        with_curvature=True, 
        with_colorbar=True,     # Hedger includes colorbars for 1D
        with_labels=False,
        with_rois=SHOW_BORDERS  # Optional borders
    )
    
    # Add the title using Matplotlib directly
    fig.suptitle(title, fontsize=16, y=0.95)
    plt.show()

def plot_2d_map(data_a, data_b, title):
    print(f"Generating 2D Integration Map: {title}...")
    vmax_a = np.nanpercentile(data_a[data_a > 0], VMAX_PERCENTILE) if np.any(data_a > 0) else 0.1
    vmax_b = np.nanpercentile(data_b[data_b > 0], VMAX_PERCENTILE) if np.any(data_b > 0) else 0.1
    
    vx = cortex.Vertex2D(
        data_a, data_b, 
        subject=cx_sub, 
        vmin=0, vmax=vmax_a, 
        vmin2=0, vmax2=vmax_b, 
        cmap=CMAP_2D
    )
    
    fig = cortex.quickflat.make_figure(
        vx, 
        with_curvature=True, 
        with_colorbar=False,    # 2D colorbars are handled externally in Hedger's workflow
        with_labels=False,
        with_rois=SHOW_BORDERS  # Optional borders
    )
    
    # Add the title using Matplotlib directly
    fig.suptitle(title, fontsize=16, y=0.95)
    plt.show()

def plot_2d_map(data_a, data_b, title):
    print(f"Generating 2D Integration Map: {title}...")
    vmax_a = np.nanpercentile(data_a[data_a > 0], VMAX_PERCENTILE) if np.any(data_a > 0) else 0.1
    vmax_b = np.nanpercentile(data_b[data_b > 0], VMAX_PERCENTILE) if np.any(data_b > 0) else 0.1
    
    vx = cortex.Vertex2D(
        data_a, data_b, 
        subject=cx_sub, 
        vmin=0, vmax=vmax_a, 
        vmin2=0, vmax2=vmax_b, 
        cmap=CMAP_2D
    )
    
    fig = cortex.quickflat.make_figure(
        vx, 
        with_curvature=True, 
        with_colorbar=False,    # 2D colorbars are handled externally in Hedger's workflow
        with_labels=False,
        with_rois=SHOW_BORDERS, # Optional borders
        title=title
    )
    plt.show()

# ==========================================
# 5. Execute Rendering
# ==========================================
plot_1d_map(r2_a_full, f"{ROI_A} R2 (Null Corrected)")
plot_1d_map(r2_b_full, f"{ROI_B} R2 (Null Corrected)")
plot_1d_map(product_full, f"{ROI_A} x {ROI_B} Product Map")

print("\n--- 2D Dual Overlay ---")
plot_2d_map(r2_a_full, r2_b_full, f"2D Integration: {ROI_A} (Color 1) vs {ROI_B} (Color 2)")

Locked onto Pycortex Anatomy: hcp_999999_draw_NH
Generating 1D Map: A5 R2 (Null Corrected)...


TypeError: make_figure() got an unexpected keyword argument 'title'